<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 21. GPU 학습·최적화 실습 — RTX 3080에서도 안전하게

이 페어 랩은 GPU를 단순히 "사용"하는 수준을 넘어, 데이터 이동·AMP·gradient accumulation,
clipping, 메모리 관찰, checkpoint/resume, 재현성까지 하나의 작은 학습 시스템으로 묶습니다.
JupyterLab 왼쪽에는 exercise, 오른쪽에는 solution을 열고 TODO를 한 셀씩 완성하세요.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 안전 원칙과 완료 기준

기본 실험은 CPU에서도 빠르게 끝나는 synthetic classification입니다. CUDA가 있으면 pinned
memory와 AMP를 자동 사용합니다. Batch-size 탐색은 **분석적 메모리 추정만 수행**해 실제 OOM을
유발하지 않습니다. RTX 3080 16GB에서도 전체 VRAM을 목표로 하지 않고 다른 앱과 CUDA context를
위해 여유를 둡니다.

완료 기준: device 확인 → 재현 가능한 loader → non-blocking 이동 → AMP 학습 → peak memory →
안전 batch 후보 → checkpoint 재개 → 조건부 `torch.compile` → 평가를 모두 통과하는 것입니다.

In [ ]:
# 준비 셀: 그대로 실행합니다.
import math
import os
import random
import tempfile
import time
from contextlib import nullcontext
from pathlib import Path

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 21
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(f"torch={torch.__version__}, {ACCELERATOR.summary()}")

## 1) Hardware inventory와 메모리 예산

코드에서 device를 문자열로 가정하지 말고 실제 가용성을 확인합니다. CUDA일 때 GPU 이름,
compute capability, 총 VRAM을 기록하세요. 총 VRAM은 곧 사용 가능 VRAM이 아닙니다. Driver,
display, 다른 프로세스, CUDA context가 함께 사용하므로 이 랩은 총량의 65%만 탐색 예산으로 둡니다.

In [ ]:
# TODO 21-1: CUDA면 get_device_properties로 이름/VRAM/capability를 읽으세요.
if device.type == "cuda":
    properties = ...
    hardware = ...
else:
    hardware = ...
print(hardware)

## 2) 재현성 설정

Python과 PyTorch RNG를 함께 고정하고, CUDA라면 모든 GPU seed를 설정합니다. Deterministic
algorithm은 느려지거나 일부 연산에서 경고가 날 수 있어 `warn_only=True`로 학습 실습을 막지
않습니다. 완전한 재현에는 데이터·코드·라이브러리·driver 버전까지 함께 기록해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,torch.manual_seed,torch.cuda.manual_seed_all,seed_everything -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

<details>
<summary><code>seed_everything(seed)</code></summary>

#### `seed_everything(seed)`

- **역할:** 프로젝트가 쓰는 여러 난수 생성기를 한 번에 고정하는 도우미입니다.
- **입력·반환:** 정수 시드를 받고 구현에 따라 설정 요약 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** Python·NumPy·PyTorch의 실험 시작 상태를 한곳에서 맞춥니다.
- **주의점:** 같은 시드만으로 모든 GPU 연산의 완전한 결정성이 보장되지는 않습니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def seed_everything(seed: int) -> None:
    # TODO 21-2: random, torch, CUDA seed와 deterministic 옵션을 설정하세요.
    raise NotImplementedError("TODO 21-2")


seed_everything(SEED)

## 3) 학습 가능한 synthetic dataset

입력 feature에 고정된 teacher matrix를 곱해 class를 만듭니다. Train/validation이 같은 teacher를
공유하므로 일반화를 측정할 수 있고, 파일 I/O 없이 GPU 파이프라인만 빠르게 반복합니다. Dataset은
CPU tensor를 반환합니다. Worker가 GPU tensor를 만드는 패턴은 CUDA context 충돌의 원인이 됩니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.randn -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class SyntheticClassificationDataset(Dataset):
    def __init__(
        self,
        size: int,
        input_dim: int,
        teacher: Tensor,
        seed: int,
    ) -> None:
        # TODO 21-3: CPU feature와 teacher 기반 label을 미리 생성하세요.
        raise NotImplementedError("TODO 21-3")

    def __len__(self) -> int:
        raise NotImplementedError("TODO 21-3")

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        raise NotImplementedError("TODO 21-3")

## 4) Windows/Jupyter에 안전한 DataLoader

CUDA일 때 `pin_memory=True`인 CPU batch를 `non_blocking=True`로 복사하면 compute와 transfer를
겹칠 가능성이 생깁니다. Windows Notebook에서는 `num_workers>0`가 셀 코드를 다시 import하거나
worker 시작 문제를 만들 수 있어 기본은 0입니다. 별도 `.py` 진입점에서만 worker 수를 조정하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataLoader -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>DataLoader(dataset, batch_size=1, shuffle=False, ...)</code></summary>

#### `DataLoader(dataset, batch_size=1, shuffle=False, ...)`

- **역할:** dataset에서 mini-batch를 반복 생성합니다.
- **입력·반환:** dataset과 batching 설정을 받고 iterable loader를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 데이터를 batch 단위로 섞고 모델에 공급합니다.
- **주의점:** Windows 다중 worker는 진입점 보호가 필요하며 마지막 batch 크기도 확인합니다.
- **공식 문서:** [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
input_dim, hidden_dim, num_classes = 128, 256, 8
teacher_generator = torch.Generator().manual_seed(999)
teacher = torch.randn(input_dim, num_classes, generator=teacher_generator)
train_dataset = SyntheticClassificationDataset(2048, input_dim, teacher, seed=101)
validation_dataset = SyntheticClassificationDataset(512, input_dim, teacher, seed=202)

batch_size = 128
loader_generator = torch.Generator().manual_seed(SEED)
# TODO 21-4: CUDA 여부에 맞춰 pin_memory를 정하고 두 DataLoader를 만드세요.
pin_memory = ...
train_loader = ...
validation_loader = ...

## 5) Host-to-device 경계 한곳에 모으기

Batch 이동을 helper 하나로 모으면 model 코드에 `.cuda()`가 흩어지지 않고 CPU fallback도
유지됩니다. `non_blocking=True`는 source가 pinned memory일 때 비동기 전송 가능성을 열어 주며,
즉시 동기화된다는 보장은 아닙니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.to -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.to(device=None, dtype=None)</code></summary>

#### `tensor.to(device=None, dtype=None)`

- **역할:** 텐서를 지정 장치나 dtype으로 변환합니다.
- **입력·반환:** 장치·dtype을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 입력의 장치·정밀도를 일치시킵니다.
- **주의점:** 원본과 조건이 같으면 복사 없이 같은 텐서를 돌려줄 수 있습니다.
- **공식 문서:** [Tensor.to](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.to.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def move_batch(batch: tuple[Tensor, Tensor]) -> tuple[Tensor, Tensor]:
    # TODO 21-5: feature/label을 device로 옮기세요. CUDA일 때 non_blocking을 사용하세요.
    raise NotImplementedError("TODO 21-5")


sample_features, sample_labels = move_batch(next(iter(train_loader)))
assert sample_features.device.type == sample_labels.device.type == device.type

## 6) 작은 모델과 sanity check

먼저 한 batch에서 forward/loss/backward shape를 확인합니다. GPU 최적화 전에 잘못된 label dtype,
output class 수, device mismatch를 잡는 가장 싼 테스트입니다. 이 모델은 의도적으로 작아 노트북을
막지 않으며, 동일한 학습 loop를 CNN/Transformer로 교체할 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,nn.GELU,nn.LayerNorm,nn.Dropout,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        # TODO 21-6: Linear-GELU-LayerNorm-Dropout-Linear를 구성하세요.
        self.network = ...

    def forward(self, features: Tensor) -> Tensor:
        return self.network(features)


model = TinyClassifier(input_dim, hidden_dim, num_classes).to(device)
parameter_count = ...

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.cross_entropy,Tensor.backward,torch.isfinite,Optimizer.zero_grad,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
sanity_logits = model(sample_features)
sanity_loss = nn.functional.cross_entropy(sanity_logits, sample_labels)
sanity_loss.backward()
assert sanity_logits.shape == (len(sample_labels), num_classes)
assert torch.isfinite(sanity_loss)
model.zero_grad(set_to_none=True)
print("sanity loss:", round(sanity_loss.item(), 4))

## 7) GPU memory 측정

`allocated`는 tensor가 실제 사용 중인 메모리, `reserved`는 allocator가 재사용하려고 확보한
메모리, `peak`는 reset 이후 최대 allocated입니다. `empty_cache()`는 살아 있는 tensor를 지우지
않으며 매 step 호출하면 오히려 느려질 수 있습니다. 측정 경계에서만 synchronize합니다.

In [ ]:
def reset_peak_memory() -> None:
    # TODO 21-7: CUDA에서 synchronize 후 peak 통계를 reset하세요.
    raise NotImplementedError("TODO 21-7")


def memory_snapshot() -> dict[str, float]:
    # TODO: CUDA allocated/reserved/peak MB를 반환하고 CPU면 0을 반환하세요.
    raise NotImplementedError("TODO 21-7")

## 8) AMP autocast + GradScaler

Autocast 영역에는 forward와 loss만 넣습니다. Backward는 autocast 밖에서 scaler를 통해 실행합니다.
Clipping 전에 반드시 `scaler.unscale_(optimizer)`를 호출해야 실제 gradient norm을 봅니다.
CPU fallback에서는 `nullcontext`와 disabled scaler로 같은 코드를 유지합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.float16 -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.float16</code></summary>

#### `torch.float16`

- **역할:** 16비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 정밀도와 메모리를 절충합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float16](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)


def amp_context():
    # TODO 21-8: CUDA면 float16 autocast, CPU면 nullcontext를 반환하세요.
    raise NotImplementedError("TODO 21-8")

## 9) Accumulation + clipping + throughput 학습 함수

Effective batch는 `micro batch × accumulation_steps`입니다. Loss를 accumulation 수로 나누고,
optimizer/scheduler/zero_grad는 update 시점에만 호출합니다. 마지막 남은 micro-batch도 update하며
clipping 전 norm과 examples/sec를 함께 기록해 안정성과 성능을 비교합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.train,perf_counter,Tensor.detach,clip_grad_norm_,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>perf_counter()</code></summary>

#### `perf_counter()`

- **역할:** 짧은 구간 측정에 적합한 고해상도 단조 시계를 읽습니다.
- **입력·반환:** 인자 없이 호출하고 초 단위 부동소수점 값을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습·추론 latency의 시작과 끝 차이를 측정합니다.
- **주의점:** 절댓값에는 의미가 없고 GPU 시간은 실행 동기화 뒤 측정해야 합니다.
- **공식 문서:** [perf_counter](https://docs.python.org/3/library/time.html#time.perf_counter)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler,
    accumulation_steps: int = 2,
    clip_norm: float = 1.0,
) -> dict[str, float]:
    # TODO 21-9: AMP/accumulation/unscale/clipping/update와 지표 집계를 구현하세요.
    raise NotImplementedError("TODO 21-9")

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.AdamW -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
first_epoch_metrics = train_one_epoch(
    model, train_loader, optimizer, scaler, accumulation_steps=2, clip_norm=1.0
)
assert first_epoch_metrics["updates"] == math.ceil(len(train_loader) / 2)
print({key: round(value, 3) for key, value in first_epoch_metrics.items()})

## 10) OOM 없는 batch-size 탐색

후보 batch를 실제로 할당하며 OOM을 잡는 코드는 allocator fragmentation과 Notebook kernel 불안정을
만들 수 있습니다. 먼저 parameter/optimizer/gradient와 activation을 보수적으로 추정해 후보를
거릅니다. 이 값은 **상한 보장치가 아닌 1차 필터**이며, 실제 모델에서는 profiler와 작은 증가
실험으로 확인하세요. Transformer activation은 sequence length 제곱 항도 포함해야 합니다.

In [ ]:
bytes_per_parameter = next(model.parameters()).element_size()
parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())


def estimate_training_bytes(candidate_batch: int) -> int:
    # TODO 21-10: weights+grads+Adam state와 보수적 activation 메모리를 추정하세요.
    raise NotImplementedError("TODO 21-10")


candidate_batches = [64, 128, 256, 512, 1024, 2048]
if device.type == "cuda":
    budget_bytes = int(torch.cuda.get_device_properties(device).total_memory * 0.65)
else:
    budget_bytes = 512 * 1024**2
safe_candidates = ...

## 11) Checkpoint 저장과 resume

Model만 저장하면 정확한 재개가 아닙니다. Optimizer, GradScaler, epoch/global step, RNG, DataLoader
generator 상태를 함께 저장합니다. 새 객체에 로드한 뒤 실제로 다음 epoch를 진행하여 resume 경로를
검증합니다. 실무에서는 scheduler, tokenizer/config, git commit, metric도 포함하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Path -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
checkpoint_path = Path(tempfile.gettempdir()) / "ai_lab21_gpu_resume_exercise.pt"
# TODO 21-11: model/optimizer/scaler/RNG/loader generator 상태를 저장하세요.
checkpoint_payload = ...
torch.save(checkpoint_payload, checkpoint_path)
assert checkpoint_path.exists()

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.cpu -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 21-12: 새 model/optimizer/scaler를 만들고 모든 상태를 복구한 뒤 한 epoch 재개하세요.
payload = ...
resumed_model = ...
resumed_optimizer = ...
resumed_scaler = ...
second_epoch_metrics = ...

## 12) `torch.compile`은 명시적 opt-in

Compile은 첫 호출 비용, graph break, backend/driver 조합 이슈가 있어 작은 Notebook에서는 오히려
느릴 수 있습니다. 기본은 꺼 두고 CUDA, API 존재, 사용자의 명시적 선택을 모두 만족할 때만
실행합니다. 먼저 eager correctness를 통과하고 고정 shape의 반복 workload에서 warmup 이후를
benchmark하세요. 문제가 나면 원본 model로 즉시 fallback합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
RUN_TORCH_COMPILE = False  # GPU에서 원할 때만 True
inference_model = resumed_model
compile_status = "eager"

# TODO 21-13: 안전 조건을 확인하고 try/except로 torch.compile을 적용하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.eval,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    start = time.perf_counter()
    correct = 0
    examples = 0
    for batch in loader:
        features, labels = move_batch(batch)
        logits = model(features)
        correct += logits.argmax(dim=1).eq(labels).sum().item()
        examples += len(labels)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    return {"accuracy": correct / examples, "examples_per_second": examples / elapsed}


final_metrics = evaluate(inference_model, validation_loader)
assert 0.0 <= final_metrics["accuracy"] <= 1.0
assert math.isfinite(final_metrics["examples_per_second"])
assert second_epoch_metrics["loss"] <= first_epoch_metrics["loss"] + 0.5
checkpoint_path.unlink(missing_ok=True)
print("validation:", {k: round(v, 3) for k, v in final_metrics.items()})
print("checkpoint cleaned:", not checkpoint_path.exists())

## 마무리 — RTX 3080 실험 체크리스트

- DataLoader worker보다 먼저 `num_workers=0 + pin_memory` 기준선을 측정합니다.
- AMP, batch size, accumulation을 한 번에 바꾸지 말고 throughput/peak VRAM/metric을 함께 기록합니다.
- `reserved > allocated`는 정상일 수 있으며 반복적인 `empty_cache()`를 최적화로 오해하지 않습니다.
- Batch-size 추정표는 1차 필터이고 sequence model은 길이 제곱 activation을 별도 반영합니다.
- Resume 테스트는 새 객체에서 수행하고 RNG/loader/scheduler/tokenizer까지 저장합니다.
- `torch.compile`은 eager 정답 검증 뒤 반복 workload에서만 opt-in benchmark합니다.

이 학습 함수를 06번 classifier, 08번 RNN, 16번 Transformer에 적용하며 모델별 peak 메모리와
examples(or tokens)/sec를 비교하면 노트북 사양에 맞는 실전 기준선을 만들 수 있습니다.